# Type 2 Diabetes — current-state phenotype inference

**Kit:** `aou_t2d_current` · **Template:** `binary_tabular` · **Prediction type:** `current_state`

> **Educational/research use only.** Educational/research output. Not a clinically validated risk score, diagnosis, or treatment recommendation.

> **SYNTHETIC DATA by default** — artificial, not clinically realistic, not derived from any
> participant data. Real All of Us data is used only inside the All of Us Researcher Workbench.

Cell banners: **Framework — do not edit** (shared infrastructure calls) · **Baseline — working
example** (copy it into your own cells to experiment) · **Your work** (runnable starter code).

## 0. Setup

Run these cells first. In Colab or local Jupyter keep `DATA_SOURCE = "synthetic"`. Inside the
**All of Us Researcher Workbench only**, a verified kit may use `DATA_SOURCE = "all_of_us"`;
anywhere else the framework refuses All of Us mode before touching any data.

In [ ]:
# Framework — do not edit
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_ml_kits") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "clinical-ml-kits @ git+https://github.com/CadenWondell/UNF-AIRO-Club.git@v0.1.0#subdirectory=projects/Fall_2026"])

In [ ]:
# Framework — do not edit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clinical_ml_kits import load_kit

DATA_SOURCE = "synthetic"  # "all_of_us" only inside the All of Us Researcher Workbench
SEED = 2101
pd.set_option("display.max_columns", 50)

kit = load_kit("aou_t2d_current", data_source=DATA_SOURCE)

## 1. Clinical problem

**Clinical question.** Given clinical information that does not define the phenotype (no diabetes diagnoses, glucose or HbA1c values, or diabetes medications), does the patient's record currently meet the type 2 diabetes phenotype?

#### Why the problem matters
Type 2 diabetes is common and often unrecognized for years; records that are consistent with the
phenotype but lack a diagnosis are a meaningful data-quality and care-gap question for research.

#### The phenotype in plain language
A person "meets the phenotype" when their record shows diabetes evidence in combination — repeated
diagnoses, a diagnosis plus treatment, or a diagnosis or treatment plus an abnormal HbA1c — and no
evidence of type 1 or another specific diabetes type. **All thresholds are provisional** pending
review of the ADA Standards of Care and the eMERGE algorithm.

#### Why this is phenotype inference, not threshold reproduction
The model never sees HbA1c, glucose, diabetes diagnoses, or diabetes medications — not even earlier
values (FR-134). It must infer the current state from other information (body size, blood pressure,
lipids, liver and renal panels, utilization), which is a genuine learning problem rather than
re-deriving the label's own rule.

#### Educational objective
Practice leakage-safe phenotype inference: understand label construction, excluded features,
index-visit bias, sampled prevalence, calibration, and honest evaluation.

> **PROVISIONAL — synthetic-only, not an approved clinical definition**
>
> The values below exist only so the kit runs on synthetic data. They are not approved clinical rules.

| Item | Provisional value | Verification |
|---|---|---|
| `cohort.inclusion[0].age_at_index.years` | `18.0` | pending V-21 |
| `temporal_policy.same_day` | `exclude_index_day` | pending V-20 |
| `temporal_policy.exclude_index_encounter` | `True` | pending V-20 |
| `temporal_policy.washout` | `P0D` | pending V-20 |
| `temporal_policy.lookback` | `P1095D` | pending V-20 |
| `phenotype.case_rule.all_of[0].any_of[0].has_event.min_count` | `2` | pending V-21 |
| `phenotype.case_rule.all_of[0].any_of[2].all_of[1].measurement.value` | `6.5` | pending V-21 |
| `phenotype.case_rule.all_of[0].any_of[3].all_of[1].measurement.value` | `6.5` | pending V-21 |
| `phenotype.control_rule.all_of[0].any_of[0].measurement.value` | `5.7` | pending V-21 |
| `phenotype.control_rule.all_of[0].any_of[1].measurement.value` | `100.0` | pending V-21 |
| `phenotype.control_sampling.controls_per_case` | `2.0` | pending V-20 |
| `phenotype.min_observation_history` | `P365D` | pending V-21 |

In [ ]:
# Framework — do not edit
print(kit.describe())

## 2. Dataset and cohort

Each step is marked **[documentation]**, **[method decision]**, **[PROVISIONAL]**, or **[deviation]**.

#### 1. Official documentation
ADA Standards of Care in Diabetes—2026, Section 2 (`ada_soc_2026_s2`, tier 4); eMERGE T2D EHR
algorithm, Kho et al. 2012 (`kho_2012_emerge_t2d`, tier 5); OMOP CDM v5.4 (`ohdsi_omop_cdm_v54`,
tier 3); All of Us Data and Statistics Dissemination Policy (`aou_dissemination_policy`, tier 1).
Primary ADA text could not be retrieved automatically; criteria are provisional until a reviewer
confirms them.

#### 2. Clinical / dataset definition
Current-state T2D phenotype (FR-130): the record is consistent with T2D at the index time.
**[documentation]** structure from eMERGE; **[PROVISIONAL]** thresholds (HbA1c ≥ 6.5 %, two
distinct-day diagnoses).

#### 3. Logical concepts and dataset realizations
`t2d_diagnosis`, `t1d_other_diabetes`, `diabetes_medications`, `hba1c`, `glucose`, plus revealer
concepts. Synthetic `SYN-` realizations exist; `omop` realizations are **empty** until verified
(V-21c, V-25).

#### 4. Cohort, cases, and controls
**[PROVISIONAL]** Adults ≥ 18. Cases: any of (≥ 2 T2D diagnoses on distinct days; diagnosis +
medication; diagnosis + HbA1c ≥ 6.5 %; medication + HbA1c ≥ 6.5 %) and no type 1/other diabetes.
Controls: HbA1c < 5.7 % or glucose < 100 mg/dL, and no diabetes diagnosis, medication, or type 1/
other code (prediabetes excluded from controls). ≥ 365 days of history for everyone.
**[method decision]** matched control sampling (ADR-002), 2 per case.

#### 5. Index and cutoff rules
**[method decision]** Case index: first date the case rule holds. Control index: one eligible
visit matched on visit type and calendar year. **[PROVISIONAL]** index day and index encounter
excluded; washout 0 days; 3-year lookback (V-20).

#### 6. Features and exclusions
See [features](features.md). Label-defining concepts and indirect revealers are prohibited.

#### 7. Kit definition (configuration)
`kit.yaml` and `phenotype.yaml` hold every rule; split salt `aou_t2d_current-s1`; seed 2101.
**[method decision]** threshold rule `youden`; AUPRC reported with prevalence.

#### 8. Notebook and baseline model
Generated Template A notebook (`kit.ipynb`): logistic regression baseline in a visible scikit-learn
`Pipeline`, grouped tuning, grouped calibration, single logged test evaluation.

The framework builds the cohort, labels, index times, features, and patient-level
train/validation/test partitions for you. The **test partition is hidden** until the final
evaluation in section 12.

In [ ]:
# Framework — do not edit
data = kit.load()
data.summary()

In [ ]:
# Baseline — working example
print("Training rows:", len(data.train.X), "| features:", data.train.X.shape[1])
data.train.X.head()

**Features used as model input**

| Feature | Meaning | Derivations |
|---|---|---|
| `age` | Age in years at the index time, derived from the person's birth date | latest |
| `sex_at_birth` | Sex at birth as recorded by the dataset (categorical) | latest |
| `bmi` | Body mass index | latest, delta, mean |
| `weight` | Body weight | latest |
| `systolic_bp` | Systolic blood pressure | latest, mean |
| `diastolic_bp` | Diastolic blood pressure | latest |
| `ldl` | LDL cholesterol | latest |
| `hdl` | HDL cholesterol | latest |
| `triglycerides` | Triglycerides | latest |
| `creatinine` | Creatinine concentration in serum/plasma | latest |
| `alt` | Alanine aminotransferase | latest |
| `hemoglobin` | Hemoglobin in blood | latest |
| `outpatient_visits` | Outpatient visit | count (window P365D) |

**Deliberately excluded (label-defining or prohibited)**

- `t2d_diagnosis` — label-defining concept
- `t1d_other_diabetes` — label-defining concept
- `diabetes_medications` — label-defining concept
- `hba1c` — label-defining concept
- `glucose` — label-defining concept
- `t2d_diagnosis` — label-defining (case evidence)
- `t1d_other_diabetes` — label-defining (exclusion evidence)
- `diabetes_medications` — label-defining (case evidence)
- `hba1c` — label-defining (case and control evidence; earlier values too, FR-134)
- `glucose` — label-defining (control evidence; earlier values too, FR-134)
- `diabetic_complications` — indirect label revealer (V-24 review)
- `glucose_monitoring` — indirect label revealer (V-24 review)
- `diabetes_education` — indirect label revealer (V-24 review)

## 3. Exploratory data analysis

Explore the training partition only. Look at missingness carefully: a missing lab is *not* a
normal lab, and how often a test was ordered can itself carry information.

In [ ]:
# Your work
missing = data.train.X.isna().mean().sort_values(ascending=False)
missing.to_frame("fraction_missing").head(20)

In [ ]:
# Your work
print("Outcome rate in training data (sampled development prevalence):",
      round(data.train.y.mean(), 3))
data.train.X.describe().T

In [ ]:
# Your work
eda_columns = ["bmi__latest", "weight__latest", "systolic_bp__latest"]
fig, axes = plt.subplots(1, len(eda_columns), figsize=(4 * len(eda_columns), 3))
for ax, col in zip(np.atleast_1d(axes), eda_columns):
    for label, part in data.train.X.groupby(data.train.y.to_numpy()):
        ax.hist(part[col].dropna(), bins=30, alpha=0.5, density=True, label=f"label={label}")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Your work
# Is missingness related to the outcome? (missing labs are informative, never "normal")
pd.DataFrame({
    "missing_rate_label_0": data.train.X[data.train.y.to_numpy() == 0].isna().mean(),
    "missing_rate_label_1": data.train.X[data.train.y.to_numpy() == 1].isna().mean(),
}).head(20)

## 4. Preprocessing

`make_preprocessor` builds the shared Template A preprocessing: median imputation **plus a
missingness indicator** for every numeric feature, then standard scaling; categorical features
get an explicit `"__missing__"` category and one-hot encoding. Inside a `Pipeline` it is fitted
on the training rows only.

In [ ]:
# Baseline — working example
from clinical_ml_kits.preprocessing import make_preprocessor

preprocessor = make_preprocessor(kit.feature_spec)
preprocessor

## 5. Baseline model

The baseline is a plain scikit-learn `Pipeline` with logistic regression. Its parameters come
from `kit.yaml`. Copy this cell into your own cells to experiment, so the baseline stays
available as a comparison point.

In [ ]:
# Baseline — working example
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("preprocess", make_preprocessor(kit.feature_spec)),
    ("model", LogisticRegression(C=1.0, penalty="l2", class_weight=None, max_iter=1000, solver="lbfgs")),
])
baseline.fit(data.train.X, data.train.y)
val_proba = baseline.predict_proba(data.validation.X)[:, 1]
val_proba[:5]

## 6. Evaluate the baseline

Expected output: a metrics table (AUROC, AUPRC with the prevalence as its baseline,
sensitivity, specificity at the `youden` threshold chosen on validation), the
confusion matrix, and ROC / precision-recall curves.

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import evaluate_binary, plot_roc_pr

baseline_results = evaluate_binary(
    data.validation.y, val_proba, threshold_rule="youden", metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"]
)
print("Confusion matrix:", baseline_results.confusion)
baseline_results.table()

In [ ]:
# Baseline — working example
plot_roc_pr(data.validation.y, val_proba, title="Baseline (validation)")

## 7. Your model

Try a different model. Use any scikit-learn estimator inside a `Pipeline` with the same
preprocessor. You never need to implement a learning algorithm yourself.

In [ ]:
# Your work
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ("preprocess", make_preprocessor(kit.feature_spec)),
    ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED)),
])
forest.fit(data.train.X, data.train.y)
forest_results = evaluate_binary(
    data.validation.y, forest.predict_proba(data.validation.X)[:, 1],
    threshold_rule="youden", metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"],
)
forest_results.table()

In [ ]:
# Your work
# Other standard estimators to try (each runs as-is; edit and compare):
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

alternatives = {
    "gradient_boosting": HistGradientBoostingClassifier(max_iter=200, random_state=SEED),
    "svm": SVC(probability=True, C=1.0, random_state=SEED),
    "knn": KNeighborsClassifier(n_neighbors=25),
}
fitted_alternatives = {}
for name, estimator in alternatives.items():
    pipe = Pipeline([("preprocess", make_preprocessor(kit.feature_spec)), ("model", estimator)])
    fitted_alternatives[name] = pipe.fit(data.train.X, data.train.y)
sorted(fitted_alternatives)

## 8. Hyperparameter tuning

Hyperparameter tuning **must** use patient-grouped folds: `cv=kit.cv_splitter()` and
`groups=data.train.groups`. Default `cv=5` is not allowed in this project.

In [ ]:
# Your work
from sklearn.model_selection import GridSearchCV

cv = kit.cv_splitter()  # patient-grouped folds within the training partition
param_grid = {"model__n_estimators": [100, 300], "model__min_samples_leaf": [2, 10]}
search = GridSearchCV(forest, param_grid, cv=cv, scoring="average_precision")
search.fit(data.train.X, data.train.y, groups=data.train.groups)
best = search.best_estimator_
print("Best parameters:", search.best_params_, "| CV AUPRC:", round(search.best_score_, 3))

## 9. Model comparison

Compare models on the **validation** partition. Never use the test partition to choose
between models.

In [ ]:
# Your work
from clinical_ml_kits.evaluation import compare_models

candidates = {"baseline": baseline, "forest": forest, "tuned_forest": best,
              **fitted_alternatives}
comparison = compare_models(candidates, data.validation, metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"],
                            threshold_rule="youden")
comparison.sort_values("auprc", ascending=False)

In [ ]:
# Your work
plot_roc_pr(
    data.validation.y,
    {name: model.predict_proba(data.validation.X)[:, 1] for name, model in candidates.items()},
    title="Validation comparison",
)

## 10. Calibration

**Sampled development prevalence vs population prevalence.** This kit samples controls
(strategy `matched`, 2.0 controls per case), so the outcome rate in the data is the *sampled development prevalence*, not the
prevalence in any real population. Predicted probabilities are calibrated (at best) to that
development sample and must not be read as a patient's population disease probability or a
clinical risk score. Recalibration uses only the grouped out-of-fold helper
`kit.calibrate_grouped` (verification item V-14).

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import calibration_summary, plot_calibration

print(calibration_summary(data.validation.y, val_proba).describe())
plot_calibration(data.validation.y, val_proba, bins=10, strategy="quantile")

In [ ]:
# Your work
# Optional recalibration: grouped out-of-fold calibration fitted on the training partition only.
calibrated = kit.calibrate_grouped(best, method="sigmoid")
cal_proba = calibrated.predict_proba(data.validation.X)[:, 1]
print(calibration_summary(data.validation.y, cal_proba).describe())

## 11. Explainability

Explain the models: standardized logistic-regression coefficients (including missingness
indicators) and permutation importance on the validation partition.

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import coefficient_table

coefficient_table(baseline).head(15)

In [ ]:
# Your work
from clinical_ml_kits.evaluation import permutation_importance_table

permutation_importance_table(best, data.validation.X, data.validation.y, seed=SEED).head(15)

## 12. Final selection and completion

Choose your final model, then run the **single** logged evaluation on the held-out test
partition. Complete the model through the standard prediction interface.

#### Sampled development prevalence vs population prevalence
Controls are sampled 2 per case (matched), so the outcome rate in the data is the **sampled
development prevalence**, not a population prevalence. Probabilities are relative to that sample and
are not population disease probabilities or clinical risk scores.

#### Index-visit bias (residual)
Matching on visit type and calendar year and excluding the index day/encounter reduce, but do not
remove, differences in why case and control visits happened. The utilization-only bias check
reports residual signal.

#### Provisional and unverified items
Every clinical threshold, temporal-policy value, and sampling ratio is provisional (V-20, V-21,
V-24, V-25); `omop` concept sets are empty. The kit stays `requires_verification` and cannot run in
the Workbench until these are verified.

#### Data and generalization limits
Synthetic data is artificial and not clinically realistic. Real EHR data has heterogeneous coding,
units, and missingness; performance on synthetic data says nothing about real performance.

In [ ]:
# Your work
final_model = best  # choose and justify your final model using validation results only

In [ ]:
# Framework — do not edit
report = kit.final_evaluation(final_model)  # held-out test partition: once, logged
print("95% bootstrap intervals on test:", report.test_ci)
report.table()

In [ ]:
# Framework — do not edit
envelopes = kit.to_envelopes(final_model, data.validation.X.head(3))
print(envelopes[0].model_dump_json(indent=2))

In [ ]:
# Framework — do not edit
import tempfile
from pathlib import Path

record = kit.complete(final_model)
print(record.model_id, "| environment:", record.environment, "| portable:", record.portable)
if kit.environment == "synthetic":  # demo models are allowed only for synthetic data
    kit.save_model(final_model, Path(tempfile.gettempdir()) / "aou_t2d_current_demo_model.joblib", portable=True)